In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
from omas import load_omas_h5
import xarray as xr
from scipy.interpolate import interp1d

from popsim import PACKAGE_ROOT
from popsim.modules.tearing import ErrorFieldLocking, TearingPhase
from popsim.simulate import SimInput, simulate, make_time_base

ds = xr.load_dataset(f"{PACKAGE_ROOT}/data/tearing/sparc_delta_per_m_n1_final_LMODE_QGT1_time11.nc")
odsSim = load_omas_h5(f"{PACKAGE_ROOT}/data/tearing/pcs_data_for_onsim2.h5")

coilnames = ds['coil_names'].values
coilsources = {}
overlaps = {}
PRD_currents = {'cs1l':38762, 
                'cs1u':38761, 
                'cs2l':313, 
                'cs2u':308, 
                'cs3l':15774, 
                'cs3u':15778, 
                'pf1l':40313,
                'pf1u':40320,
                'pf2l':41117,
                'pf2u':41122,
                'pf3l':34242,
                'pf3u':34232,
                'pf4l':37002,
                'pf4u':37000,
                'tfjumpers':31250,
                'div1u':30000,
                'div1l':30000,
                'div2u':30000,
                'div2l':30000,
                'divu':30000,
                'divl':30000,
                'om4tflare_upper':45000,
                'om4tflare_lower':45000}

for i in ds['coil_index'].values:
    coil = str(ds['coil_names'][i].values).upper()
    if any(['CS' in coil, 'PF' in coil, 'TF' in coil]):
        coilsources[coil] = ['shift', 'tilt', 'nominal']
        overlaps[coil] = {}
        overlaps[coil]['shift'] = ds['delta_per_m_displacement'][i].values/1000*(1.+0.00000001j)/PRD_currents[coil.lower()]
        overlaps[coil]['tilt'] = ds['delta_per_m_tilt'][i].values/1000*(1.+0.00000001j)/PRD_currents[coil.lower()]
        overlaps[coil]['nominal'] = 0.*(1.+0.00000001j)/PRD_currents[coil.lower()] # Not included in the dummy data

    else:
        coilsources[coil] = ['nominal']
        overlaps[coil] = {}
        overlaps[coil]['nominal'] = 0.*(1.+0.00000001j)/PRD_currents[coil.lower()]

overlaps['TF'] = {} # Since the coils don't have 
overlaps['TF']['shift'] = 1e-5*(1.+0.00000001j)/3.12e6
overlaps['TF']['tilt'] = 1e-5*(1.+0.00000001j)/3.12e6
overlaps['TF']['nominal'] = 0.*(1.+0.00000001j)/3.12e6
coilsources['TF'] = ['shift', 'tilt', 'nominal']
for name in ['DIV1U','DIV1L','DIV2U','DIV2L']:
    overlaps[name] = {}
    overlaps[name]['shift'] = 0.*(1.+0.00000001j)/PRD_currents[name.lower()]
    overlaps[name]['tilt'] = 0.*(1.+0.00000001j)/PRD_currents[name.lower()]
    overlaps[name]['nominal'] = 0.*(1.+0.00000001j)/PRD_currents[name.lower()]
    coilsources[name] = ['tilt', 'shift', 'nominal']

# This is dummy overlap data at the moment; will need a 
# file that contains the final overlap #s from Nik. Meeting about this on 05/31/2024.

coilsources['VSC'] = []

metrology = {}
for key1 in overlaps.keys():
    metrology[key1] = {}
    for key2 in overlaps[key1].keys():
        if key2 in ['tilt', 'shift']:
            metrology[key1][key2] = 1/np.sqrt(2) *( np.random.random() + 1.j* np.random.random()) # random 0-1 mm, random phase tilt/shift for all coils
        else:
            metrology[key1][key2] = 1.

modes = [(2, 1)]
error_field_locking_config = ErrorFieldLocking.Config(
    modes = modes,
    metrology=metrology,
    overlaps=overlaps,
    static_sources={},
    hysteresis=0.9,
    efc_efficiency=0.5,
)

error_field_locking_initial_state = ErrorFieldLocking.State(
    W={mode: 0.0 for mode in modes}, 
    F={mode: 0.0 for mode in modes}, 
    mode_phase={mode: 0.0 for mode in modes},
    tearing_phase={mode: TearingPhase.NONE for mode in modes},
)

scaling_laws = json.load(open(f"{PACKAGE_ROOT}/data/tearing/scalinglaws.json"))
scaling_law_terms = scaling_laws["O,L:WLS"]

popsim_time_base = make_time_base(t0=0.0, t1=20, dt=0.1)

ods_time_base = odsSim['summary.time']

ods_time_trace = {
    "coeff": 10.,
    "Bt": 12.2,
    "R": 1.85,
    "ne": odsSim['summary.line_average.n_e.value']/1e19,
    "beta_N": odsSim['summary.global_quantities.beta_tor_norm.value'],
    "Ip": odsSim['summary.global_quantities.ip.value'],
    "li": odsSim['summary.global_quantities.li.value'],
}

# For each value in the ods time base, resample the ods time trace to the popsim time base

scaling_law_params = {}
for key, value in ods_time_trace.items():
    if np.isscalar(value):
        scaling_law_params[key] = np.full_like(popsim_time_base, value)
    else:
        interp_function = interp1d(ods_time_base, value, kind="previous")
        scaling_law_params[key] = interp_function(popsim_time_base)

error_field_locking_params = ErrorFieldLocking.Params(
    scaling_law_terms=scaling_law_terms,
    scaling_law_params=scaling_law_params,
)

error_field_locking_module = ErrorFieldLocking(config=error_field_locking_config)

sim_input = SimInput(time=popsim_time_base, initial_state=error_field_locking_initial_state, params=error_field_locking_params)

sim_xarray = simulate(module=error_field_locking_module, sim_inputs=sim_input)
